
## THis is the Silver layer for our Project


Reading the Bronze tables

In [0]:
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA retailpulse_project")

In [0]:
customers_bronze_df = spark.table(
    "workspace.retailpulse_project.bronze_customers"
)

products_bronze_df = spark.table(
    "workspace.retailpulse_project.bronze_products"
)

orders_bronze_df = spark.table(
    "workspace.retailpulse_project.bronze_orders"
)

In [0]:
customers_bronze_df.printSchema()
products_bronze_df.printSchema()
orders_bronze_df.printSchema()

1. Create Silver Customers

In [0]:
from pyspark.sql.functions import col

silver_customers_df = (
    customers_bronze_df
    .withColumn(
        "customer_id",
        col("customer_id").cast("int")
    )
)

In [0]:
silver_customers_df.printSchema()

In [0]:
# Dropping the Extra ingestion_columns from 2 tables & Keep only the orders timestamp

customers_clean = silver_customers_df.drop("ingestion_timestamp")


Save Silver Customers in Delta Table (silver)

In [0]:
silver_customers_df.write \
    .format("delta") \
        .mode("overwrite") \
            .saveAsTable(
                "workspace.retailpulse_project.silver_customers"
            )

2. Create Silver Products

In [0]:
silver_products_df = (
    products_bronze_df
    .withColumn(
        "product_id",
        col("product_id").cast("int")
    )
    .withColumn(
        "price",
        col("price").cast("double")
    )
)

In [0]:
# Dropping the Extra ingestion_columns from 2 tables & Keep only the orders timestamp

products_clean = silver_products_df.drop("ingestion_timestamp")

In [0]:
#Save as Table

silver_products_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.retailpulse_project.silver_products"
    )


3. Create Silver Orders


In [0]:

silver_orders_df = (
    orders_bronze_df
    .withColumn(
        "order_id",
        col("order_id").cast("int")
    )
    .withColumn(
        "customer_id",
        col("customer_id").cast("int")
    )
    .withColumn(
        "product_id",
        col("product_id").cast("int")
    )
    .withColumn(
        "quantity",
        col("quantity").cast("int")
    )
    .withColumn(
        "order_date",
        col("order_date").cast("date")
    )
)

In [0]:
# Save as table

silver_orders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.retailpulse_project.silver_orders"
    )


### Verify the Schemas

In [0]:
# Joining happened

In [0]:
silver_sales_enriched_df = (
    silver_orders_df
        .join(
            customers_clean,
            on="customer_id",
            how="inner"
        )
        .join(
            products_clean,
            on="product_id",
            how="inner"
        )
)

In [0]:
silver_sales_enriched_df.printSchema()

In [0]:
# Add revenue Column

silver_sales_enriched_df = (
    silver_sales_enriched_df.withColumn(
        "revenue",
        col("quantity") * col("price")
    )
)

In [0]:
# Load the datas into the delta table

silver_sales_enriched_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.retailpulse_project.silver_sales_enriched"
    )

In [0]:
display(silver_sales_enriched_df)

In [0]:
spark.table(
    "workspace.retailpulse_project.silver_sales_enriched"
).show()